In [1]:
import os
import rasterio
import numpy as np
import subprocess
import pandas as pd
import glob
from osgeo import gdal

def compute_stats(summer_vrt, winter_vrt):
    """Compute summary statistics for summer vs winter rasters by band."""
    stats = []

    with rasterio.open(summer_vrt) as src_summer, rasterio.open(winter_vrt) as src_winter:
        if src_summer.count != src_winter.count:
            raise ValueError("Summer and Winter rasters must have the same number of bands.")
        if src_summer.width != src_winter.width or src_summer.height != src_winter.height:
            raise ValueError("Summer and Winter rasters must have the same dimensions.")

        for band in range(1, src_summer.count + 1):
            summer_data = src_summer.read(band).astype(float)
            winter_data = src_winter.read(band).astype(float)

            # Mask NaNs
            mask = ~np.isnan(summer_data) & ~np.isnan(winter_data)
            summer_vals = summer_data[mask]
            winter_vals = winter_data[mask]

            # Compute stats
            mean_diff = np.mean(summer_vals - winter_vals)
            pct_change = np.mean((summer_vals - winter_vals) / winter_vals * 100)

            stats.append({
                "Band": band,
                "Summer Mean": np.mean(summer_vals),
                "Winter Mean": np.mean(winter_vals),
                "Mean Diff": mean_diff,
                "Mean % Change": pct_change,
                "Summer Min": np.min(summer_vals),
                "Summer Max": np.max(summer_vals),
                "Winter Min": np.min(winter_vals),
                "Winter Max": np.max(winter_vals),
            })

    return pd.DataFrame(stats)

In [2]:
def build_mosaic_vrt_in_memory(raster_dir, pattern, tp):
    # Find the two input rasters
    rasters = sorted(glob.glob(os.path.join(raster_dir, pattern)))
    if len(rasters) < 2:
        raise ValueError(f"Expected >=2 rasters matching {pattern}, found {len(rasters)}")

    # Build an in-memory VRT (mosaic)
    # Use separate=False so overlapping areas are blended; you can also set options like 'resolution', 'resampling'
    vrt_path = f"/vsimem/{tp}_mosaic.vrt"
    vrt = gdal.BuildVRT(
        vrt_path,
        rasters,
        separate=False,        # combine as a single band mosaic (not band stack)
        # resolution='highest', # or 'lowest' or 'user' with 'xRes','yRes'
        # resampling='nearest', # 'bilinear','cubic', etc., if different grids
    )
    if vrt is None:
        raise RuntimeError("BuildVRT failed.")

    # Flush to ensure the VRT XML is written into /vsimem
    vrt.FlushCache()
    return vrt_path

In [5]:
raster_path = '/mnt/f/readyparams/ppp_paramsoutput/anaxyrus_americanus/'
# Example usage:
#summer_files = build_mosaic_vrt_in_memory(raster_path, pattern="*summer*.tif", tp='summer')
#winter_files = build_mosaic_vrt_in_memory(raster_path, pattern="*winter*.tif", tp='winter')

dfstats = compute_stats(os.path.join(raster_path, 'predictions_prob_anaxyrus_americanus_summer.tif'), os.path.join(raster_path, 'predictions_prob_anaxyrus_americanus_winter.tif'))
print(dfstats)

   Band  Summer Mean  Winter Mean  Mean Diff  Mean % Change    Summer Min  \
0     1     0.005849     0.007958  -0.002109     108.966675  5.095909e-25   

   Summer Max    Winter Min  Winter Max  
0         1.0  1.449382e-18         1.0  


In [6]:
display(dfstats)

,Band,Summer Mean,Winter Mean,Mean Diff,Mean % Change,Summer Min,Summer Max,Winter Min,Winter Max
0,1,0.005849,0.007958,-0.002109,108.966675,5.095909e-25,1.0,1.449382e-18,1.0
